In [31]:
import numpy as np
import cvxpy as cp
import mosek
import random
import matplotlib.pyplot as plt
from itertools import chain, combinations

In [32]:
def h_3(x,alfa):
    return(min(1,x/(1-alfa)))

def powerset(iterable):
    "powerset([1,2,3]) --> () (1,) (2,) (3,) (1,2) (1,3) (2,3) (1,2,3)"
    s = list(iterable)
    return chain.from_iterable(combinations(s, r) for r in range(len(s)+1))

def ranktoset (A):
    A = list(A)
    sets = [[A[0]]]
    for i in range(1,len(A)):
        new = A[0:i+1]
        sets.append(new)
    return(sets)

def makesetflex (A, B):    # we assume A is non-empty
    N = len(A)
    B = list(B)
    M = len(B)
    added = []
    for i in range(M):
        new = B[0:i+1]
        for k in range(N):
            if len(A[k])==len(new) and len(np.intersect1d(A[k],new))==len(new):
                break
            if k == N-1:
                A.append(new)
                added.append(new)
    return(A,added)

def countsets(sets):
    m = len(sets)
    count = 0
    for k in range(m):
        count = count + len(sets[k])
    return(count)

def convertlist(sets):
    Output = []
    for temp in sets:
        for elem in temp:
            Output.append(elem)
    return(Output)


In [33]:
def robustcheckpowerU(a,R,r,c,p,m,r_f,rav):    ### actually not being used
    N = len(p)
    x = -(R.dot(a)+(1-sum(a))*r_f)**rav/rav
    rank = np.argsort(-x)
    extra = 0
    if np.min(x) < 0:
        extra = np.min(x)
        c = c - np.min(x)
        x = x - np.min(x)
    q_b = cp.Variable(N, nonneg = True)
    q = cp.Variable(N, nonneg=True)
    constraints = [cp.sum(q) == 1]
    phi_cons = 0
    for i in range(N):
        z1 = q_b[rank[0:i+1]]
        z2 = q[rank[0:i+1]]
        v = -cp.neg(cp.sum(z2)/(1-m)-1)+1
        constraints.append(cp.sum(z1)-v <= 0)
        phi_cons = phi_cons -cp.entr(q[i]) - q[i]*np.log(p[i])
    constraints.append(phi_cons <= r)
    obj = cp.Maximize(q_b.T @ x)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(prob.value<=c)

def riskcalc(a,R,p,alfa,r_f):
    x = -R.dot(a)
    rank = np.argsort(R.dot(a))
    extra = 0
    if np.min(x) < 0:
        extra = np.min(x)
        x = x - np.min(x)
    N = len(p)
    risk = h_3(p[rank[0]],alfa)*x[rank[0]]
    for i in range(2,N+1):
        z1 = sum(p[rank[0:i]])
        z2 = sum(p[rank[0:i-1]])
        risk = risk + (h_3(z1,alfa)-h_3(z2,alfa))*x[rank[i-1]]
    risk = risk + extra - (1-sum(a))*r_f
    print("the nominal risk of a:", risk)

In [67]:
def solvenominalpowerU (sets,p,R,r,m,r_f,c,rav):
    N = len(p)
    I = len(R[0])
    M = len(sets)
    v = cp.Variable((M,N))
    lbda = cp.Variable(M, nonneg = True)
    a = cp.Variable(I)
    alpha = cp.Variable(1)
    beta = cp.Variable(1)
    gamma = cp.Variable(1,nonneg = True)
    t = cp.Variable(N, nonneg = True)
    z2 = 0
    z4 = 0
    f_obj = 0
    constraints = []
    for i in range(N):
        lbdasum = 0
        for j in range(M):
            if i in sets[j]:
                constraints.append(v[j][i] >= 0)
                lbdasum = lbdasum + lbda[j]
            else:
                constraints.append(v[j][i] >= 0)
        z3 = (R @ a)[i]+(1-cp.sum(a))*r_f
        f_obj = cp.power(z3,rav)/rav*p[i] + f_obj
        constraints.append((-R @ a)[i] - lbdasum - beta <= 0)
        z4 = z4 + p[i]*t[i]
        constraints.append(-alpha + cp.sum(v[0:M:1,i]) + cp.kl_div(gamma, t[i]) + gamma - t[i] <= 0)
    for j in range(M):
        z1 = -cp.min(v[j,sets[j]])*(1-m)+lbda[j]
        z2 = z2 + cp.pos(z1)
    constraints.append(a<= 1)
    constraints.append(a>=0)
    constraints.append(cp.sum(a)<=1)
    constraints.append(alpha + beta + gamma * (r-1)- (1-cp.sum(a))*r_f + z4 + z2 <= c)
    obj = cp.Maximize(f_obj)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(a.value, prob.value)
    
def robustcheck(a,R,r,c,p,m,r_f):
    N = len(p)
    x = -R.dot(a)
    rank = np.argsort(R.dot(a))
    extra = 0
    if np.min(x) < 0:
        extra = np.min(x)
        c = c - np.min(x)
        x = x - np.min(x)
    q_b = cp.Variable(N, nonneg = True)
    q = cp.Variable(N, nonneg=True)
    constraints = [cp.sum(q) == 1]
    phi_cons = 0
    for i in range(N):
        z1 = q_b[rank[0:i+1]]
        z2 = q[rank[0:i+1]]
        v = -cp.neg(cp.sum(z2)/(1-m)-1)+1
        constraints.append(cp.sum(z1)-v <= 0)
        phi_cons = phi_cons -cp.entr(q[i]) - q[i]*np.log(p[i])
    constraints.append(phi_cons <= r)
    obj = cp.Maximize(q_b.T @ x)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    print(prob.value - (1-np.sum(a))*r_f)
    print(c)
    return(prob.value - (1-np.sum(a))*r_f <= c)  




In [39]:
def cutting_plane(R,r,c,p,m,r_f,sets,rav):
    nonstop = True
    iterations = 1
    while nonstop == True:
        #print(solvenominalpowerU(realsets,p,R,r,m,r_f,c,rav))
        [a,obj] = solvenominalpowerU(sets,p,R,r,m,r_f,c,rav)
        newrank = np.argsort(R.dot(a))
        [sets,added] = makesetflex(sets, newrank)
        if robustcheck(a,R,r,c,p,m,r_f) == True:
            return(a,obj,iterations)
        iterations = iterations + 1
    

In [36]:
np.random.seed(5)

In [63]:
N=5
p = (np.zeros(N)+1)*1/N
I = 1
R = np.random.normal(0.05,0.2,size=(N,I))
print(R.transpose().dot(p))
print(R)

[0.01227202]
[[ 0.3326796 ]
 [ 0.09425082]
 [-0.21215463]
 [-0.08791305]
 [-0.06550265]]


In [69]:
gam = 1.1
rav = 1-gam
r = 1
m = 0.05    # this is the parameter of the h function min(1, p/(1-m))
r_f = 0.01
c = 0.1
sets = [[0]]
result = cutting_plane(R,r,c,p,m,r_f,sets,rav)
print(result)

-0.009679599092418195
0.10019690076649869
(array([0.00059186]), -15.848825662089663, 1)


In [45]:
psets = list(powerset(list(range(N))))
for i in range(1,len(psets)):
    psets[i] = list(psets[i])
psets = psets[1:(len(psets))]
print(solvenominalpowerU (psets,p,R,r,m,r_f,c,rav))
#print(robustcheckpowerU(a,R,r,c,p,m,r_f,rav))


(array([2.12246020e-01, 6.57178201e-10, 1.74291315e-09, 1.36559227e-01]), 0.4017701842173073)


In [46]:
sets

[[0], [1], [1, 4], [1, 4, 3], [1, 4, 3, 2], [1, 4, 3, 2, 0]]

In [36]:
def phi_div(p,q,r):
    phi_cons = 0
    for i in range(len(p)):
        phi_cons = q[i]*np.log(q[i]/p[i])+phi_cons
    print(phi_cons <= r)
    print(phi_cons)

In [128]:
a = result[0]
(R.dot(a)+(1-sum(a))*r_f)

array([0.00324602, 0.10003987, 0.0017258 , 0.01083311, 0.01283602])

In [129]:
a[3]= a[3]+0.01
(R.dot(a)+(1-sum(a))*r_f)

array([0.00023825, 0.09970399, 0.00317519, 0.00998637, 0.01590632])

In [116]:
((R.dot(a)+(1-sum(a))*r_f)**rav/rav).dot(p)

0.25159836997179363

In [130]:
robustcheck(a,R,r,c,p,m,r_f)

0.09245733106379603
0.5930596311633911


True

In [115]:
a

array([ 4.60655049e-10, -2.70685378e-10,  1.16719833e-01,  4.01829532e-02,
        1.78661803e-01])

In [51]:
def tempsolve(p,R,rav,r_f):
    N = len(p)
    I = len(R[0])
    a = cp.Variable(I)
    constraints = [0<=a, a<=1, cp.sum(a)<=1]
    f_obj = 0
    for i in range(N):
        z3 = (R @ a)[i]+(1-cp.sum(a))*r_f
        f_obj = cp.power(z3,rav)/rav*p[i] + f_obj
    obj = cp.Maximize(f_obj)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(a.value, prob.value)
    

In [62]:
tempsolve(p,R,rav,r_f)

(array([4.73368797e-01, 2.97327108e-01, 2.65846612e-09, 2.29304091e-01]),
 0.709226061087395)